# Automotive: Durability Cycle Comparison

Compare stress and load data across multiple durability test cycles to assess structural fatigue and identify anomalous runs.

**Context:** Automotive durability testing subjects components to repeated stress cycles. Each session represents one test run. Engineers compare runs to detect drift, early fatigue, or anomalous loading patterns.

In [ ]:
import sys
sys.path.insert(0, '..')
from sqlrace_helpers import (
    init_sqlrace, load_session, session_summary,
    list_parameters, extract_parameter, extract_to_timetable
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import os
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")

# Durability test sessions (one per cycle)
CYCLES = [
    ("53f2367d-14b6-43d1-aeaf-6e66f1b1031c", "durability-cycle1.ssn2"),
    ("f8d31ef8-48ba-450b-971c-8fec68dec7f8", "durability-cycle2.ssn2"),
    ("54c2a62c-08c5-4f35-9a85-8a276ec83329", "durability-cycle3.ssn2"),
]

# Primary stress/load parameter
STRESS_PARAM = "Force:Actuator01"

sm = init_sqlrace()

## Load and extract each cycle

In [ ]:
cycle_data = {}
cycle_stats = []

for i, (guid, filename) in enumerate(CYCLES):
    cs = f"DbEngine=SQLite;Data Source={os.path.join(DATA_DIR, filename)};"
    client, sess = load_session(sm, guid, connection_string=cs)
    try:
        data = extract_parameter(sess, STRESS_PARAM)
        # Convert to relative seconds
        t0 = data.index[0]
        rel_time = (data.index - t0) / 1e9
        cycle_data[f"Cycle {i+1}"] = pd.Series(data.values, index=rel_time)

        cycle_stats.append({
            "Cycle": i + 1,
            "Samples": len(data),
            "Duration (s)": rel_time[-1],
            "Mean": data.mean(),
            "Std": data.std(),
            "Peak (abs)": data.abs().max(),
            "RMS": np.sqrt((data.values ** 2).mean()),
        })
        print(f"  Cycle {i+1}: {len(data)} samples")
    finally:
        client.Dispose()

df_stats = pd.DataFrame(cycle_stats).set_index("Cycle")
display(df_stats)

## Overlay cycle traces

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for name, data in cycle_data.items():
    ax.plot(data.index, data.values, label=name,
            linewidth=0.6, alpha=0.7)

ax.set_xlabel("Time (s)")
ax.set_ylabel(STRESS_PARAM)
ax.set_title("Durability Cycle Overlay")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Peak distribution

Compare the distribution of peak loads across cycles using histograms.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, data in cycle_data.items():
    ax.hist(data.values, bins=50, alpha=0.5, label=name, density=True)

ax.set_xlabel(STRESS_PARAM)
ax.set_ylabel("Density")
ax.set_title("Load Distribution Comparison")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## RMS trend across cycles

RMS (root mean square) of the stress signal is a proxy for fatigue damage accumulation.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_stats.index, df_stats["RMS"], 'o-', markersize=8, linewidth=2)
ax.set_xlabel("Cycle")
ax.set_ylabel("RMS Load")
ax.set_title("RMS Trend Across Cycles")
ax.grid(True, alpha=0.3)

# Flag cycles where RMS deviates >10% from first cycle
baseline = df_stats["RMS"].iloc[0]
for idx, rms in df_stats["RMS"].items():
    if abs(rms - baseline) / baseline > 0.10:
        ax.annotate(f"{(rms-baseline)/baseline:+.1%}",
                    (idx, rms), textcoords="offset points",
                    xytext=(0, 12), ha='center', color='red')

plt.tight_layout()
plt.show()

## Anomaly detection

Flag cycles where key metrics deviate significantly from the group.

In [ ]:
print("Anomaly Check (>2 sigma from group mean)")
print("=" * 50)

for metric in ["Mean", "Std", "Peak (abs)", "RMS"]:
    group_mean = df_stats[metric].mean()
    group_std = df_stats[metric].std()

    for cycle, val in df_stats[metric].items():
        if group_std > 0 and abs(val - group_mean) > 2 * group_std:
            z = (val - group_mean) / group_std
            print(f"  Cycle {cycle}: {metric} = {val:.4f} "
                  f"(z = {z:+.2f}, group mean = {group_mean:.4f})")

print("\nDone.")